## Projected Exit Dashboard Sandbox
- Owner: Hanna Lee, DHS
- Created On: 3/6/2026
- Last Modified On: 3/6/2026

In [2]:
## import modules 
import pandas as pd
import numpy as np
import os
from datetime import datetime as dt

# Import SQL connection modules + others. 
import pyodbc

In [3]:
def get_database():

    global startdate, enddate, updated_time
        
    ## Connection parameters.
    server = 'BKODS01SQL16' ## it could be 'localhost' for wnidows connection. 
    database = 'StreetSolutions'
    driver = '{ODBC Driver 18 for SQL Server}'

    # Create connection string using Windows Authentication
    conn_str = f'DRIVER={driver};SERVER={server};Trusted_Connection=yes; TrustServerCertificate=yes'

    # Establish connection
    conn = pyodbc.connect(conn_str)

    ## Queries to pull based on neeeds
    # Site Compliance
    pmoc_query = f'''
                    SELECT *
                    FROM StreetSolutions.dbo.SHS_Weekly_PMO_Compliance_vw

        '''
    # Monthly PMOs with Additional Info    
    pmo_query = f'''--CREATE OR ALTER VIEW SHS_Weekly_PMO_Additional_Info_vw as

                    WITH weekly_pmo as (SELECT v1.ID,
                        v1.ResponderName, 
                        v1.Dateofsubmission, 
                        v1.FacilityType, 
                        v1.SiteName, 
                        v2.Facility_CD, 
                        v2.PA,
                        v1.SiteHasPMO, 
                        v1.ClientFirstName, 
                        v1.ClientLastName,
                        TRY_CAST(REPLACE(v1.CARESId, ' ', '') AS INT) AS CARESID_int,
                        v1.StreetSmartId,
                        v1.TargetMoveOutDate, 
                        v1.Exit_Reason, 
                        v1.MOProgress_Detail, 
                        v1.CaseProvider, 
                        v1.StartOfWeek, 
                        v1.EndOfWeek, 
                        v1.WeekPeriod, 
                        v1.DayofSubmission, 
                        v1.Late_Submission_FLG
                    FROM StreetSolutions.dbo.SHS_Weekly_PMO_Stg v1
                    LEFT JOIN StreetSolutions.dbo.SHS_ProjectedMoveOut_CodeTable v2 on v1.siteName = v2.List_of_Sites_SHS_Portfolio
                    ), 
                    table_v1 as (SELECT t1.*,
                        t2.Facility_Name, 
                        t2.Facility_CD as Facility_CD_LP, 
                        t2.Placement_Start_DTTM, 
                        t2.Exit_DTTM, 
                        t2.Exit_Reason as Exit_Reason_LP,
                        CASE WHEN (EXISTS (SELECT t2.Facility_CD
                                            FROM StreetSolutions.dbo.SHSFacility_Dec25 t3
                                            WHERE t2.Facility_CD = t3.Facility_CD
                                            ) OR (t2.Facility_CD IS NULL))
                                            THEN 'N/A'
                                            ELSE 'Y'
                                        END AS Latest_Exit_Elsewhere_FLG
                    FROM weekly_pmo t1
                    LEFT JOIN StreetSolutions.dbo.Latest_EDW_Placements_Fct t2 on t1.CARESID_int = t2.CARES_ID
                    ), 
                    table_v2 as (SELECT *, 
                    CASE WHEN Latest_Exit_Elsewhere_FLG = 'Y'
                    AND Placement_Start_DTTM > DateofSubmission 
                    THEN 'Y'
                    ELSE 'N'
                    END AS Reentered_system_FLG, 
                    CASE WHEN Latest_Exit_Elsewhere_FLG = 'Y'
                    AND FacilityType NOT IN ('Drop-in Center', 'Outreach Team')
                    THEN 'Y'
                    ELSE 'N'
                    END AS Error_FLG
                    FROM table_v1 
                    ), 
                    additional_info as (SELECT ID, 
                        Latest_Exit_Elsewhere_FLG, 
                        Reentered_system_FLG, 
                        Error_FLG
                    FROM table_v2
                    )
                    SELECT *
                    FROM table_v2
                    ORDER BY DateofSubmission DESC

        '''
    
    all_pmo = '''WITH weekly_pmo as (SELECT v1.ID,
                        v1.ResponderName, 
                        v1.Dateofsubmission, 
                        v1.FacilityType, 
                        v1.SiteName, 
                        v2.Facility_CD, 
                        v2.PA,
                        v1.SiteHasPMO, 
                        v1.ClientFirstName, 
                        v1.ClientLastName,
                        TRY_CAST(REPLACE(v1.CARESId, ' ', '') AS INT) AS CARESID_int,
                        v1.StreetSmartId,
                        v1.TargetMoveOutDate, 
                        v1.Exit_Reason, 
                        v1.MOProgress_Detail, 
                        v1.CaseProvider, 
                        v1.StartOfWeek, 
                        v1.EndOfWeek, 
                        v1.WeekPeriod, 
                        v1.DayofSubmission, 
                        v1.Late_Submission_FLG
                    FROM StreetSolutions.dbo.SHS_Weekly_PMO_Stg v1
                    LEFT JOIN StreetSolutions.dbo.SHS_ProjectedMoveOut_CodeTable v2 on v1.siteName = v2.List_of_Sites_SHS_Portfolio
                    ), 
                    table_v1 as (SELECT t1.*,
                        t2.Facility_Name, 
                        t2.Facility_CD as Facility_CD_LP, 
                        t2.Placement_Start_DTTM, 
                        t2.Exit_DTTM, 
                        t2.Exit_Reason as Exit_Reason_LP,
                        CASE WHEN (EXISTS (SELECT t2.Facility_CD
                                            FROM StreetSolutions.dbo.SHSFacility_Dec25 t3
                                            WHERE t2.Facility_CD = t3.Facility_CD
                                            ) OR (t2.Facility_CD IS NULL))
                                            THEN 'N/A'
                                            ELSE 'Y'
                                        END AS Latest_Exit_Elsewhere_FLG
                    FROM weekly_pmo t1
                    LEFT JOIN StreetSolutions.dbo.Latest_EDW_Placements_Fct t2 on t1.CARESID_int = t2.CARES_ID
                    ), 
                    table_v2 as (SELECT *, 
                    CASE WHEN Latest_Exit_Elsewhere_FLG = 'Y'
                    AND Placement_Start_DTTM > DateofSubmission 
                    THEN 'Y'
                    ELSE 'N'
                    END AS Reentered_system_FLG, 
                    CASE WHEN Latest_Exit_Elsewhere_FLG = 'Y'
                    AND FacilityType NOT IN ('Drop-in Center', 'Outreach Team')
                    THEN 'Y'
                    ELSE 'N'
                    END AS Error_FLG
                    FROM table_v1 
                    ), 
                    additional_info as (SELECT ID, 
                        Latest_Exit_Elsewhere_FLG, 
                        Reentered_system_FLG, 
                        Error_FLG
                    FROM table_v2
                    )
                    SELECT *
                    FROM table_v2
                    WHERE Reentered_System_FLG = 'Y'
                    ORDER BY DateofSubmission DESC

        '''
    
    ## Reading from SQL database the three datasets.
    global pmodf, cpdf, reentry
    pmodf = pd.read_sql(pmo_query, conn)
    cpdf = pd.read_sql(pmoc_query, conn)
    reentry = pd.read_sql(all_pmo, conn)
    return(len(pmodf), len(cpdf))

In [4]:
get_database()

C:\Users\hanlee.DHS\AppData\Local\Temp\ipykernel_31032\241580287.py:159: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pmodf = pd.read_sql(pmo_query, conn)
C:\Users\hanlee.DHS\AppData\Local\Temp\ipykernel_31032\241580287.py:160: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cpdf = pd.read_sql(pmoc_query, conn)
C:\Users\hanlee.DHS\AppData\Local\Temp\ipykernel_31032\241580287.py:161: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reentry = pd.read_sql(all_pmo, conn)


(2621, 3304)

In [5]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=BKODS01SQL16;"
    "DATABASE=StreetSolutions;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes"
)

In [6]:
import os
from dotenv import load_dotenv

load_dotenv('venv\\.env')
## Connection parameters 
server = os.getenv("DB_SERVER")
driver = os.getenv("DB_DRIVER")
## Create connection string using Windows autentification. 
conn_str = f'DRIVER={driver};SERVER={server};Trusted_Connection=yes; TrustServerCertificate=yes'
conn_str

'DRIVER={ODBC Driver 18 for SQL Server};SERVER=BKODS01SQL16;Trusted_Connection=yes; TrustServerCertificate=yes'

In [7]:
os.getenv("DB_SERVER")

'BKODS01SQL16'

In [8]:
print(server)

BKODS01SQL16


In [9]:
import pandas as pd

In [10]:
pd.read_csv('data\\compliance.csv')

,Unnamed: 0,WeekPeriod,StartOfWeek,EndOfWeek,List_of_Sites_SHS_Portfolio,Facility_Type,PA,Facility_CD,Report_Status,Late_Submission_FLG
0,0,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,105 STREET SAFE HAVEN,Safe Haven,Brenita,M011,Y,Y
1,1,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,BRC BOWERY SAFE HAVEN,Safe Haven,Mohammed,M034,N,N
2,2,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,BREAKING GROUND SEAFARERS,Safe Haven,Mohammed,M019,Y,Y
3,3,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,BronxWorks Outreach Team,Outreach Team,Elena,NaN,N,N
4,4,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,CARE FOUND HERE MORRIS AVENUE,Safe Haven,Ernest,XS07,N,N
...,...,...,...,...,...,...,...,...,...,...
3299,3299,2026-03-07 - 2026-03-13,2026-03-07,2026-03-13,MOC Outreach Team,Outreach Team,Elena,NaN,N,N
3300,3300,2026-03-07 - 2026-03-13,2026-03-07,2026-03-13,OLD BROADWAY SAFE HAVEN,Safe Haven,Ernest,M022,N,N
3301,3301,2026-03-07 - 2026-03-13,2026-03-07,2026-03-13,QUEENS DROP IN CENTER,Drop-in Center,Cindy,DI06,Y,N
3302,3302,2026-03-07 - 2026-03-13,2026-03-07,2026-03-13,THE ANDREWS SAFE HAVEN,Safe Haven,Mohammed,H074,N,N


In [30]:
from pathlib import Path
from datetime import datetime

file_path = Path("data\\compliance.csv")
last_modified = datetime.fromtimestamp(file_path.stat().st_mtime)

print(last_modified)

2026-03-09 16:07:16.856189


In [33]:
str(last_modified)[:10]

'2026-03-09'

In [44]:
compliance = pd.read_csv('data\\compliance.csv')
pmo_data = pd.read_csv('data\\pmo_data.csv')
reentry_data = pd.read_csv('data\\reentry_data.csv')

In [49]:
test = compliance[(compliance['PA']=='Mohammed') & (compliance['Facility_Type']=='Drop-in Center')]

In [52]:
pd.pivot_table(compliance, index='List_of_Sites_SHS_Portfolio',
                          columns='Late_Submission_FLG', aggfunc='count')[['WeekPeriod']].reset_index()

List_of_Sites_SHS_Portfolio WeekPeriod      
Late_Submission_FLG                                                    N     Y
0                                       105 STREET SAFE HAVEN       56.0   3.0
1                               127TH STREET KELLY SAFE HAVEN       56.0   3.0
2                                   9TH AVENUE DROP-IN CENTER       54.0   5.0
3                                           BG AT 83RD STREET       53.0   6.0
4                                       BRC BOWERY SAFE HAVEN       55.0   4.0
5                                               BRC RECEPTION       54.0   5.0
6                                           BRC Outreach Team       52.0   7.0
7                                   BREAKING GROUND SEAFARERS       57.0   2.0
8                                BREAKING GROUND WILLIAMSBURG       53.0   6.0
9                         BRONXWORKS SAFE HAVEN - LIVING ROOM       53.0   6.0
10                     Breaking Ground Brooklyn Outreach Team       49.0  10.0
11                       Breaking Ground Queens Outreach Team       52.0   7.0
12                                   BronxWorks Outreach Team       51.0   8.0
13                              CARE FOUND HERE MORRIS AVENUE       56.0   3.0
14                                 CARPENTER HOUSE SAFE HAVEN       57.0   2.0
15                      COMUNILIFE E. 146 STREET SAFE HAVEN I       59.0   NaN
16                   COMUNILIFE SAFE HAVEN II (MONROE AVENUE)       44.0  15.0
17                                     CONTINENTAL SAFE HAVEN       53.0   6.0
18                                 CROMWELL AVENUE SAFE HAVEN       57.0   2.0
19                                   EAST FLATBUSH SAFE HAVEN       57.0   2.0
20                                     EAST HARLEM SAFE HAVEN       58.0   1.0
21                                     EASTSIDE STABILIZATION       59.0   NaN
22                                         HEGEMAN SAFE HAVEN       59.0   NaN
23                                     INFINITI STABILIZATION       58.0   1.0
24                                 LIVING ROOM DROP-IN CENTER       52.0   7.0
25                             LONG ACRE STABILIZATION - CUCS       48.0  11.0
26                     LONG ACRE STABILIZATION -URBAN PATHWAY       56.0   3.0
27                              MAINCHANCE/GCN DROP-IN CENTER       50.0   9.0
28                                  MARMION AVENUE SAFE HAVEN       58.0   1.0
29                                         MIDWOOD SAFE HAVEN       55.0   4.0
30                                          MOC Outreach Team       54.0   5.0
31                                    OLD BROADWAY SAFE HAVEN       59.0   NaN
32                                    OLIVIERI DROP IN CENTER       59.0   NaN
33                                PARK OVERLOOK STABILIZATION       56.0   3.0
34                                     PARKSIDE STABILIZATION       58.0   1.0
35                                       PAUL'S PLACE DROP IN       56.0   3.0
36                                    PAUL'S PLACE SAFE HAVEN       54.0   5.0
37                                PROJECT HOSPITALITY DROP-IN       51.0   8.0
38                        PROSPECT STABILIZATION BEDS PROGRAM       48.0  11.0
39                                         PYRAMID SAFE HAVEN       59.0   NaN
40                          Project Hospitality Outreach Team       45.0  14.0
41                                      QUEENS DROP IN CENTER       53.0   6.0
42                                     STEBBINS STABILIZATION       58.0   1.0
43                                     THE ANDREWS SAFE HAVEN       55.0   4.0
44                                      THE BAXTER SAFE HAVEN       56.0   3.0
45                                        THE GATHERING PLACE       49.0  10.0
46                                    THE HOLLY STABILIZATION       57.0   2.0
47                             TOV STABILIZATION BEDS PROGRAM       55.0   4.0
48                                      TRAVELER'S SAFE HAVEN       59.0   NaN
49   

In [51]:
test.sort_values(by='StartOfWeek')

,Unnamed: 0,WeekPeriod,StartOfWeek,EndOfWeek,List_of_Sites_SHS_Portfolio,Facility_Type,PA,Facility_CD,Report_Status,Late_Submission_FLG
5,5,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,N,N
89,89,2025-02-01 - 2025-02-07,2025-02-01,2025-02-07,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,N,N
221,221,2025-02-08 - 2025-02-14,2025-02-08,2025-02-14,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,N,N
26,26,2025-02-15 - 2025-02-21,2025-02-15,2025-02-21,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,N,N
235,235,2025-02-22 - 2025-02-28,2025-02-22,2025-02-28,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,Y,N
601,601,2025-03-01 - 2025-03-07,2025-03-01,2025-03-07,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,Y,Y
303,303,2025-03-08 - 2025-03-14,2025-03-08,2025-03-14,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,N,N
615,615,2025-03-15 - 2025-03-21,2025-03-15,2025-03-21,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,Y,Y
445,445,2025-03-22 - 2025-03-28,2025-03-22,2025-03-28,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,Y,Y
453,453,2025-03-29 - 2025-04-04,2025-03-29,2025-04-04,MAINCHANCE/GCN DROP-IN CENTER,Drop-in Center,Mohammed,DI02,N,N


In [46]:
test

,Unnamed: 0,WeekPeriod,StartOfWeek,EndOfWeek,List_of_Sites_SHS_Portfolio,Facility_Type,PA,Facility_CD,Report_Status,Late_Submission_FLG


In [37]:
len(test)

0

In [40]:
if test.empty: 
    print("Empty")

Empty


In [99]:
compliance

,Unnamed: 0,WeekPeriod,StartOfWeek,EndOfWeek,List_of_Sites_SHS_Portfolio,Facility_Type,PA,Facility_CD,Report_Status,Late_Submission_FLG
0,0,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,105 STREET SAFE HAVEN,Safe Haven,Brenita,M011,Y,Y
1,1,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,BRC BOWERY SAFE HAVEN,Safe Haven,Mohammed,M034,N,N
2,2,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,BREAKING GROUND SEAFARERS,Safe Haven,Mohammed,M019,Y,Y
3,3,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,BronxWorks Outreach Team,Outreach Team,Elena,NaN,N,N
4,4,2025-01-25 - 2025-01-31,2025-01-25,2025-01-31,CARE FOUND HERE MORRIS AVENUE,Safe Haven,Ernest,XS07,N,N
...,...,...,...,...,...,...,...,...,...,...
3299,3299,2026-03-07 - 2026-03-13,2026-03-07,2026-03-13,MOC Outreach Team,Outreach Team,Elena,NaN,N,N
3300,3300,2026-03-07 - 2026-03-13,2026-03-07,2026-03-13,OLD BROADWAY SAFE HAVEN,Safe Haven,Ernest,M022,N,N
3301,3301,2026-03-07 - 2026-03-13,2026-03-07,2026-03-13,QUEENS DROP IN CENTER,Drop-in Center,Cindy,DI06,Y,N
3302,3302,2026-03-07 - 2026-03-13,2026-03-07,2026-03-13,THE ANDREWS SAFE HAVEN,Safe Haven,Mohammed,H074,N,N


In [103]:
compliance['Late_Submission_FLG'].unique()

array(['Y', 'N'], dtype=object)

In [104]:
test = compliance[(compliance['PA']=='Brenita') & (compliance['Facility_Type']=='Stabilization')].sort_values(by='WeekPeriod')

In [106]:
test.tail(30)['Late_Submission_FLG'].unique()

array(['N'], dtype=object)

In [19]:
facility_dict = build_reporting_tables(data)


In [20]:
import streamlit as st

In [21]:
tab_map = {"Safe Haven":"Safe Haven", "Stabilization":"Stabilization", "Drop-in Center":"Drop-in Center", "Outreach Team ":"Outreach Team "}

In [22]:
tab_name = 'Stabilization'

In [23]:
get_table_height(facility_dict.get(tab_map[tab_name]))

420

In [25]:
data.columns

Index(['Unnamed: 0', 'WeekPeriod', 'StartOfWeek', 'EndOfWeek',
       'List_of_Sites_SHS_Portfolio', 'Facility_Type', 'PA', 'Facility_CD',
       'Report_Status', 'Late_Submission_FLG'],
      dtype='object')

In [27]:
compliance.columns

Index(['Unnamed: 0', 'WeekPeriod', 'StartOfWeek', 'EndOfWeek',
       'List_of_Sites_SHS_Portfolio', 'Facility_Type', 'PA', 'Facility_CD',
       'Report_Status', 'Late_Submission_FLG'],
      dtype='object')